In [14]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
import argparse
import os
import random
import argparse
from karateclub import NetLSD, Graph2Vec, FeatherGraph
from grakel import GraphKernel
import toper
import networkx as nx

from sklearn.manifold import MDS
from sklearn.decomposition import PCA

from magnipy.magnitude.distances import get_dist

from diversity_curves.process import get_mag_graphs, get_funs_df, get_mag_funs_graphs
from diversity_curves.simulate import simulate_graphs_kwargs
from diversity_curves.classifiers import knn_classifier
from diversity_curves.clustering_performance import compute_clustering_scores_resample, compute_silhouette_score_resamples, compute_rand_agglo, compute_rand_kmeans, all_clustering_results_euclidean
from diversity_curves.baselines.spectral_zoo import est_moment
from diversity_curves.baselines.ph import obtain_input_from_graphs

# scratch space script

In [16]:

# argparser = argparse.ArgumentParser()
# argparser.add_argument("--output_dir", type=str, default="./simulations/results/sim_params/", help="Output directory for results")
# argparser.add_argument('--models', type=str, nargs='*', default=None, help='select the models to be computed, has to be a subset of the models specifies in models_configs')
# argparser.add_argument('--ngraphs', type=int, default=3, help='Number of graphs that are generated per cardinality')
# argparser.add_argument('--nrange', type=int, nargs=2, default=[10,30], help='specify the range of cardinalities of the generated graphs')
# argparser.add_argument('--nsteps', type=int, default=1, help='Specify the stepsize for the cardinalities of the generated graphs')
# argparser.add_argument('--nseed', type=int, default=10, help='specify the number of different seeds for the ablation study')

# args = argparser.parse_args()
# results_dir = args.output_dir
n_low = 10
n_high = 30
n_steps = 1
n_graphs = 3
n_seeds = 1

# setting the parameters as reported in the paper
models_configs = {
'ER': {'ER_probs': [0.1, 0.5, 0.9], 
    'ER_nps': None}, 
'SBM': {
    'SBM_probs': [0.6, 0.01], 
    'SBM_blocks': [[1/2], [1/3, 1/3], [1/4, 1/4, 1/4]] 
    },
'RG': {
    'RG_probs': [0.25, 0.5, 0.75]
},
'RPG':{
    'RPG_probs': [[0.9, 0.01], [0.9, 0.1], [0.9, 0.5]],
    'RPG_props': [1/3, 1/3] 
},
}

models = list(models_configs.keys())

# if args.models is None:
#     models = models = list(models_configs.keys())
# else:
#     models = args.models

# choice of graph metric we use for the spread computation
graph_metrics = ['shortest_path_distance']

scores = ['acc_mean', 'silh_mean', 'ari_agglo_mean', 'db_mean',	'ch_mean', 'ari_kmeans_mean']
scores_asc = {'acc_mean': False, 'silh_mean': False, 'ari_agglo_mean': False, 'db_mean': True,'ch_mean': False, 'ari_kmeans_mean': False}

# results_dir = f'{output_dir}/nodes_{n_low}_to_{n_high}_stepsize_{n_steps}/n_graphs_{n_graphs}/'

# if not os.path.exists(results_dir):
#     os.makedirs(results_dir)


all_configs = {}
all_results = {}


model = models[0]

model_results = {}
scores_dfs = []
model_params = models_configs[model].keys()
config = [models_configs[model][para] for para in model_params]
model_kwargs = models_configs[model]

# setting parameters and folder for plots
sns.set_style('white')
matplotlib.rcParams.update({'font.size': 18})
# if not os.path.exists(results_dir + f'{model}/'):
#     os.makedirs(results_dir + f'{model}/')

# generate graphs
G_nx, Gs, y, y_probs, n_nodes = simulate_graphs_kwargs(
        model = model, n_graphs=n_graphs,
        nodes_range=range(n_low,n_high, n_steps),
        **model_kwargs,
        )

# labels for PCA plots
match model:
    case 'ER':
        hue_names = {str(i): f'$p={p}$' for i, p in enumerate(model_kwargs['ER_probs'])}
    case 'RG':
        hue_names = {str(i): f'$r={r}$' for i, r in enumerate(model_kwargs['RG_probs'])}
    case 'SBM':
        hue_names = {
            str(0): '$2$ communities',
            str(1): '$3$ communities',
            str(2): '$4$ communities'}
    case 'RPG':
        hue_names = {str(i): f'$p_{{in}}={p[0]}, p_{{out}}={p[1]}$' for i, p in enumerate(model_kwargs['RPG_probs'])}

compute_at_cardinality = None
upsample_to = max(n_nodes) 
downsample_to = None


##################################
### computing diversity curves ###
##################################

for metric in graph_metrics:

    acc_per_seed = {}
    acc_per_seed[f'divcurves_{metric}'] = []
    acc_per_seed[f'divcurves_pca_{metric}'] = []

    mag_funs_per_seed = []

    model_results[f'divcurves_{metric}'] = {}
    model_results[f'divcurves_pca_{metric}'] = {}
    model_results["divcurves_"+metric]['acc_mean'] = []
    model_results["divcurves_"+metric]['acc_std'] = []
    model_results[f'divcurves_pca_{metric}']['acc_mean'] = []
    model_results[f'divcurves_pca_{metric}']['acc_std'] = []

    for seed in range(n_seeds):
        np.random.seed(seed)
        random.seed(seed)

        # model_results[f'div_function_{metric}'] = {}

        print(f"Computing MAG with metric: {metric}")

        def dist_fn(G):
            return get_dist(G=G, X=None, X2=None, metric=metric, mode="structure")
        
        all_spreads, weights_lists = get_mag_graphs([g.copy() for g in G_nx], dist_fn, ts=[1], upsample_to=upsample_to, downsample_to=downsample_to, n_repeats=1, scores_method='random', compute_at_cardinality=compute_at_cardinality)

        mag_funs = get_funs_df(all_spreads)
        mag_funs_interpolated = mag_funs.interpolate(method='linear', axis='columns')

        mag_funs_per_seed.append(mag_funs_interpolated)

        # # compute clustering metrics based on euclidean distance of diversity curve vectors 
        # ari_kmeans_mean, ari_kmeans_std = compute_rand_kmeans(np.array(mag_funs_interpolated), y)
        # ari_agglo_mean, ari_agglo_std = compute_rand_agglo(np.array(mag_funs_interpolated), y, metric='euclidean')

        # means, stds = compute_clustering_scores_resample(np.array(mag_funs_interpolated), y)

        # model_results[f'divcurves_{metric}']['silh_mean'], model_results[f'divcurves_{metric}']['db_mean'], model_results[f'divcurves_{metric}']['ch_mean'] = means
        # model_results[f'divcurves_{metric}']['silh_std'], model_results[f'divcurves_{metric}']['db_std'], model_results[f'divcurves_{metric}']['ch_std'] = stds
        # model_results[f'divcurves_{metric}']['ari_agglo_mean'] = ari_agglo_mean
        # model_results[f'divcurves_{metric}']['ari_agglo_std'] = ari_agglo_std
        # model_results[f'divcurves_{metric}']['ari_kmeans_mean'] = ari_kmeans_mean
        # model_results[f'divcurves_{metric}']['ari_kmeans_std'] = ari_kmeans_std

        # computing accuracies
        if n_graphs > 1:
            test_accs = knn_classifier(np.array(mag_funs_interpolated), np.array(y), precomputed=False, groups=np.array(n_nodes))
        else:
            test_accs = knn_classifier(np.array(mag_funs_interpolated), np.array(y), precomputed=False)
        
        acc_per_seed["divcurves_"+metric].append(test_accs)
        model_results["divcurves_"+metric]['acc_mean'].append(np.mean(test_accs))
        model_results["divcurves_"+metric]['acc_std'].append(np.std(test_accs))

        # doing PCA with Euclidean distance between the diversity curves
        pca_embedding = PCA(n_components=2)
        X_pca = pca_embedding.fit_transform(np.array(mag_funs_interpolated))
        
        # computing clustering scores and accuracies on transformed data from PCA
        # all_clustering_results_euclidean(f'divcurves_pca_{metric}', model_results, X_pca, y)
        if n_graphs > 1:
            test_accs = knn_classifier(X_pca, np.array(y), precomputed=False, groups=np.array(n_nodes))
        else:
            test_accs = knn_classifier(X_pca, np.array(y), precomputed=False)
        
        acc_per_seed[f'divcurves_pca_{metric}'].append(test_accs)
        model_results[f'divcurves_pca_{metric}']['acc_mean'].append(np.mean(test_accs))
        model_results[f'divcurves_pca_{metric}']['acc_std'].append(np.std(test_accs))

        # #creating PCA plots
        # fig = sns.scatterplot(x=X_pca[:,0], y=X_pca[:,1], hue=y, palette='tab10')
        # handles, labels = fig.get_legend_handles_labels()
        # fig.legend(
        #     handles=handles,
        #     labels= [hue_names[old_label] for old_label in labels],
        #     frameon=False,
        #     loc = (1.04, 0),
        #     bbox_to_anchor=(1, 0.5),
        # )
        # sns.despine()
        # plt.savefig(results_dir + f"{model}/divcurves_pca_{metric}.pdf", dpi=300, format='pdf', bbox_inches="tight")
        # plt.savefig(results_dir + f"{model}/divcurves_pca_{metric}.png", dpi=300, format='png', bbox_inches="tight")
        # plt.close()



###########################
### computing baselines ###
###########################

        struct_baselines = {
            'nnodes': lambda G: G.number_of_nodes(),
            'nedges': lambda G: G.number_of_edges(),
            'density': lambda G: nx.density(G),
            'nconn_components': lambda G: nx.number_connected_components(G),
            'avg_clustering': lambda G: nx.average_clustering(G),
            'girth': lambda G: nx.girth(G),
        }


        # for collecting a summary statistic vector
        sum_stat = []

        # one at a time
        for n_baseline in struct_baselines:
            array = np.reshape(np.array([struct_baselines[n_baseline](G_nx[i]) for i in range(len(G_nx))]), (-1,1))

            model_results[f'strucure_baseline_{n_baseline}'] = {}
            model_results[f'strucure_baseline_{n_baseline}'] = {}
            model_results[f'strucure_baseline_{n_baseline}'] = {}

            all_clustering_results_euclidean(f'strucure_baseline_{n_baseline}', model_results, array, y) 
            
            if n_graphs > 1:
                test_accs = knn_classifier(array, np.array(y), precomputed=False, groups=np.array(n_nodes))
            else:
                test_accs = knn_classifier(array, np.array(y), precomputed=False)
            model_results[f'strucure_baseline_{n_baseline}']['acc_mean'] = np.mean(test_accs)
            model_results[f'strucure_baseline_{n_baseline}']['acc_std'] = np.std(test_accs)

            sum_stat.append(array)
        
        # summary statistic
        array_sum_stat = np.concatenate([a for a in sum_stat], axis=1)
        
        model_results[f'strucure_baseline_nnodes'] = {}
        model_results[f'strucure_baseline_nnodes'] = {}
        model_results[f'strucure_baseline_nnodes'] = {}

        all_clustering_results_euclidean(f'strucure_baseline_nnodes', model_results, array_sum_stat, y) 
        
        if n_graphs > 1:
            test_accs = knn_classifier(array_sum_stat, np.array(y), precomputed=False, groups=np.array(n_nodes))
        else:
            test_accs = knn_classifier(array_sum_stat, np.array(y), precomputed=False)
        model_results[f'strucure_baseline_nnodes']['acc_mean'] = np.mean(test_accs)
        model_results[f'strucure_baseline_nnodes']['acc_std'] = np.std(test_accs)

# storing results of model
res_df = pd.DataFrame.from_dict(model_results, orient='index')
all_results[model] = res_df
all_configs[model] = str(config)



Computing MAG with metric: shortest_path_distance
Fold 0: test accuracy = 1.0000 (k=5)
Fold 1: test accuracy = 1.0000 (k=5)
Fold 2: test accuracy = 1.0000 (k=5)
Fold 3: test accuracy = 1.0000 (k=5)
Fold 4: test accuracy = 1.0000 (k=5)
Fold 5: test accuracy = 1.0000 (k=5)
Fold 6: test accuracy = 1.0000 (k=5)
Fold 7: test accuracy = 1.0000 (k=5)
Fold 8: test accuracy = 1.0000 (k=5)
Fold 9: test accuracy = 1.0000 (k=5)

Mean test accuracy: 1.0000
Std test accuracy:  0.0000
Fold 0: test accuracy = 1.0000 (k=5)
Fold 1: test accuracy = 1.0000 (k=5)
Fold 2: test accuracy = 1.0000 (k=5)
Fold 3: test accuracy = 1.0000 (k=5)
Fold 4: test accuracy = 1.0000 (k=5)
Fold 5: test accuracy = 1.0000 (k=5)
Fold 6: test accuracy = 1.0000 (k=5)
Fold 7: test accuracy = 1.0000 (k=5)
Fold 8: test accuracy = 1.0000 (k=5)
Fold 9: test accuracy = 1.0000 (k=5)

Mean test accuracy: 1.0000
Std test accuracy:  0.0000
Silhouette Score: -0.059 ± 0.009
Davies-Bouldin Score: 132.609 ± 575.517
Calinski-Harabasz Score: 0.

ValueError: Input X contains infinity or a value too large for dtype('float64').

In [18]:
array

array([[inf],
       [inf],
       [inf],
       [ 3.],
       [ 3.],
       [inf],
       [ 3.],
       [ 4.],
       [inf],
       [inf],
       [inf],
       [ 3.],
       [inf],
       [inf],
       [ 3.],
       [ 3.],
       [ 3.],
       [inf],
       [ 4.],
       [ 5.],
       [ 5.],
       [inf],
       [ 3.],
       [ 3.],
       [inf],
       [inf],
       [ 3.],
       [ 5.],
       [ 3.],
       [inf],
       [ 3.],
       [ 4.],
       [ 3.],
       [ 3.],
       [ 4.],
       [ 3.],
       [ 4.],
       [ 5.],
       [ 3.],
       [ 3.],
       [ 3.],
       [ 3.],
       [ 3.],
       [ 3.],
       [ 3.],
       [ 4.],
       [ 3.],
       [ 3.],
       [ 3.],
       [ 3.],
       [ 3.],
       [ 3.],
       [ 3.],
       [ 3.],
       [ 3.],
       [ 3.],
       [ 3.],
       [ 3.],
       [ 3.],
       [ 3.],
       [ 3.],
       [ 3.],
       [ 3.],
       [ 3.],
       [ 3.],
       [ 3.],
       [ 3.],
       [ 3.],
       [ 3.],
       [ 3.],
       [ 3.],
      

In [54]:
res_df

,acc_mean,acc_std,silh_mean,db_mean,ch_mean,silh_std,db_std,ch_std,ari_agglo_mean,ari_agglo_std,ari_kmeans_mean,ari_kmeans_std
divcurves_shortest_path_distance,[1.0],[0.0],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
divcurves_pca_shortest_path_distance,[1.0],[0.0],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
strucure_baseline_nnodes,0.922222,0.066667,0.181196,1.428044,42.169082,0.040582,0.257139,7.252736,0.166423,0.078092,0.221464,0.061702
strucure_baseline_nedges,0.716667,0.145827,0.181238,1.416510,42.478445,0.042661,0.257429,7.314445,0.168162,0.084161,0.221517,0.061492
strucure_baseline_density,1.0,0.0,0.910079,0.128194,5112.601068,0.006217,0.009778,830.348690,1.000000,0.000000,1.000000,0.000000
strucure_baseline_nconn_components,0.4,0.133333,0.035302,0.540263,63.096574,0.029742,0.061205,14.823324,0.210267,0.118062,0.272736,0.060799


In [46]:
alist = [array1, array2]

In [51]:
np.concatenate([a for a in alist], axis=1)

array([[1, 2],
       [2, 3],
       [2, 3]])

In [41]:
array1 = np.array([[1],[2],[2]])
array2 = np.array([[2],[3],[3]])
np.concatenate([array1, array2], axis=1)

array([[1, 2],
       [2, 3],
       [2, 3]])

In [42]:
array3 = array1
array3 = np.concatenate([array3, array2], axis=1)

In [45]:
array2

array([[2],
       [3],
       [3]])

In [43]:
array3

array([[1, 2],
       [2, 3],
       [2, 3]])

In [35]:
reshaped = np.reshape(array1, (-1,1))

In [39]:
reshaped.shape

(180, 1)

In [37]:
array1

array([10, 10, 10, 11, 11, 11, 12, 12, 12, 13, 13, 13, 14, 14, 14, 15, 15,
       15, 16, 16, 16, 17, 17, 17, 18, 18, 18, 19, 19, 19, 20, 20, 20, 21,
       21, 21, 22, 22, 22, 23, 23, 23, 24, 24, 24, 25, 25, 25, 26, 26, 26,
       27, 27, 27, 28, 28, 28, 29, 29, 29, 10, 10, 10, 11, 11, 11, 12, 12,
       12, 13, 13, 13, 14, 14, 14, 15, 15, 15, 16, 16, 16, 17, 17, 17, 18,
       18, 18, 19, 19, 19, 20, 20, 20, 21, 21, 21, 22, 22, 22, 23, 23, 23,
       24, 24, 24, 25, 25, 25, 26, 26, 26, 27, 27, 27, 28, 28, 28, 29, 29,
       29, 10, 10, 10, 11, 11, 11, 12, 12, 12, 13, 13, 13, 14, 14, 14, 15,
       15, 15, 16, 16, 16, 17, 17, 17, 18, 18, 18, 19, 19, 19, 20, 20, 20,
       21, 21, 21, 22, 22, 22, 23, 23, 23, 24, 24, 24, 25, 25, 25, 26, 26,
       26, 27, 27, 27, 28, 28, 28, 29, 29, 29])

In [28]:
array1 = np.array([G_nx[i].number_of_nodes() for i in range(len(G_nx))])

In [12]:
struct_baselines = {
    'nnodes': lambda G: G.number_of_nodes(),
    'nedges': lambda G: G.number_of_edges(),
    'density': lambda G: nx.density(G),
    'nconn_components': lambda G: nx.number_connected_components(G),
    'avg_clustering': lambda G: nx.average_clustering(G),
    'girth': lambda G: nx.girth(G),
}

In [13]:
np.reshape(np.array([struct_baselines['avg_clustering'](G_nx[i]) for i in range(len(G_nx))]), (-1,1))

NameError: name 'G_nx' is not defined

# results

### Generating Tables 

In [23]:
# read in all results and configurations
resdir = 'simulations/results/ablations/params/structural_baseline'
res_df = pd.read_csv(f'{resdir}/results_combined.csv', header=[0,1], index_col=0)

configs = pd.read_csv(f'{resdir}/configs.csv', header=[0], index_col=0)

scores = ['acc', 'silh', 'db', 'ch', 'ari_agglo', 'ari_kmeans']
scores_asc = {'acc': False, 'silh': False, 'ari_agglo': False, 'db': True,'ch': False, 'ari_kmeans': False}

# group methods
method_group = {}
for method in list(res_df.index):
    if 'structure_baseline' in method:
        method_group[method] = 'Baseline'
    elif 'divcurves' in method:
        method_group[method] = 'Diversity Curves'

res_df['group'] = [method_group[method] for method in list(res_df.index)]

#rename rows, only relevant ones, rest will be dropped
new_names = {
        'divcurves_shortest_path_distance': 'Shortest Path',
        'divcurves_pca_shortest_path_distance': 'Shortest Path PCA',
        'structure_baseline_nnodes': 'Number of Nodes',
        'structure_baseline_nedges': 'Number of Edges',
        'structure_baseline_density': 'Density',
        'structure_baseline_nconn_components': 'Number of Connected Components',
        'structure_baseline_avg_clustering': 'Clustering Coefficient',
        'structure_baseline_summary': 'Summary Statistic'
}


df = res_df.rename(new_names)
df = df.loc[list(new_names.values()),:]

# add group to multiindex
df = df.set_index('group', append = True).swaplevel(0,1)
df.index.rename(['Group', 'Method'], inplace = True)


# create table for Latex
results_latex_dfs = {}
models = list(np.unique(df.columns.get_level_values(0)))

for model in models:
    results_latex_model = {}
    # sort wrt to accuracy, in groups
    df_model = df.groupby(level=0, sort=False).apply(lambda x: x.sort_values((model, 'acc_mean'), ascending=False)).droplevel(0)
    res_copy = df_model.copy()
    for score in scores:
        mean_col = (model, f'{score}_mean')
        std_col = (model, f'{score}_std')
        if mean_col in res_copy.columns and std_col in res_copy.columns:
            # formatting of the latex table
            results_latex_model[score] = res_copy[mean_col].combine(
                res_copy[std_col],
                lambda m, s: '{:.3f}'.format(round(m, 3)) + ' $\pm $ ' + '{:.3f}'.format(round(s, 3))
            )
    results_latex_df_model = pd.DataFrame.from_dict(results_latex_model)

# Bold the best mean for each score
    for score in scores:
        mean_col = (model, f"{score}_mean")
        display_col = (model, score)
        if mean_col in res_copy.columns:
            if not scores_asc[score]: 
                best_mask = round(res_copy[mean_col],3) == round(res_copy[mean_col].max(),3)
            else:
                best_mask = round(res_copy[mean_col],3) == round(res_copy[mean_col].min(),3)
            results_latex_df_model.loc[best_mask, score] = results_latex_df_model.loc[best_mask, score].apply(
                lambda x: f"\\textbf{{{x}}}"
            )
        else:
            print(f'{mean_col} not in dataframe')

    #final adjustments latex df 
    results_latex_df_model.index.rename([None, 'Method'], inplace = True)
    new_cols = {
        'acc': 'Accuracy', 
        'silh': 'Silhouette Score', 
        'db': 'Davies-Bouldin', 
        'ch': 'Calinski-Harabasz', 
        'ari_kmeans': 'ARI k-means', 
        'ari_agglo': 'ARI agglomerative'
    }

    results_latex_df_model = results_latex_df_model.rename(columns=new_cols)
    results_latex_df_model.replace(to_replace='nan $\pm $ nan', value='-', inplace=True)

    #store df in dict
    results_latex_dfs[model] = results_latex_df_model

# print table for each model
for model in models:
    print(f'model: {model} with config' + configs.loc[(model, 'configurations')])
    print(results_latex_dfs[model].to_latex(multirow=False, escape = False))

model: ER with config[[0.1, 0.5, 0.9], None]
\begin{tabular}{llllllll}
\toprule
         &                 &                     Accuracy &             Silhouette Score &               Davies-Bouldin &                 Calinski-Harabasz &            ARI agglomerative &                  ARI k-means \\
{} & Method &                              &                              &                              &                                   &                              &                              \\
\midrule
Diversity Curves & Shortest Path &  \textbf{1.000 $\pm $ 0.000} &           0.684 $\pm $ 0.020 &           0.341 $\pm $ 0.018 &             277.144 $\pm $ 56.035 &           0.482 $\pm $ 0.048 &           0.440 $\pm $ 0.018 \\
         & Shortest Path PCA &  \textbf{1.000 $\pm $ 0.000} &           0.696 $\pm $ 0.020 &           0.324 $\pm $ 0.019 &             281.461 $\pm $ 57.475 &           0.483 $\pm $ 0.044 &           0.441 $\pm $ 0.016 \\
Baseline & Density &  \textbf{1.00

In [22]:
results_latex_df_model

Accuracy  \
                 Method                                                        
Diversity Curves Shortest Path                            0.861 $\pm $ 0.051   
                 Shortest Path PCA                        0.856 $\pm $ 0.083   
Baseline         Density                         \textbf{0.922 $\pm $ 0.067}   
                 Summary Statistic                        0.778 $\pm $ 0.117   
                 Number of Connected Components           0.583 $\pm $ 0.112   
                 Clustering Coefficient                   0.556 $\pm $ 0.124   
                 Number of Edges                          0.494 $\pm $ 0.128   
                 Number of Nodes                          0.333 $\pm $ 0.000   

                                                            Silhouette Score  \
                 Method                                                        
Diversity Curves Shortest Path                            0.285 $\pm $ 0.035   
                 Shortest Path PCA                        0.293 $\pm $ 0.036   
Baseline         Density                         \textbf{0.538 $\pm $ 0.038}   
                 Summary Statistic                       -0.051 $\pm $ 0.018   
                 Number of Connected Components           0.047 $\pm $ 0.038   
                 Clustering Coefficient                  -0.035 $\pm $ 0.027   
                 Number of Edges                         -0.066 $\pm $ 0.020   
                 Number of Nodes                         -0.059 $\pm $ 0.009   

                                                              Davies-Bouldin  \
                 Method                                                        
Diversity Curves Shortest Path                            1.078 $\pm $ 0.117   
                 Shortest Path PCA                        1.052 $\pm $ 0.115   
Baseline         Density                         \textbf{0.573 $\pm $ 0.064}   
                 Summary Statistic                        3.689 $\pm $ 1.030   
                 Number of Connected Components           1.940 $\pm $ 0.308   
                 Clustering Coefficient                   4.217 $\pm $ 3.775   
                 Number of Edges                          3.622 $\pm $ 1.158   
                 Number of Nodes                      132.609 $\pm $ 575.517   

                                                              Calinski-Harabasz  \
                 Method                                                           
Diversity Curves Shortest Path                             97.448 $\pm $ 15.504   
                 Shortest Path PCA                         99.781 $\pm $ 16.010   
Baseline         Density                         \textbf{306.839 $\pm $ 39.498}   
                 Summary Statistic                          10.941 $\pm $ 3.791   
                 Number of Connected Components             25.349 $\pm $ 5.717   
                 Clustering Coefficient                     11.432 $\pm $ 2.911   
                 Number of Edges                            11.481 $\pm $ 3.991   
                 Number of Nodes                             0.550 $\pm $ 0.612   

                                                           ARI agglomerative  \
                 Method                                                        
Diversity Curves Shortest Path                            0.384 $\pm $ 0.078   
                 Shortest Path PCA                        0.378 $\pm $ 0.078   
Baseline         Density                         \textbf{0.535 $\pm $ 0.076}   
                 Summary Statistic                        0.040 $\pm $ 0.027   
                 Number of Connected Components           0.108 $\pm $ 0.066   
                 Clustering Coefficient                   0.027 $\pm $ 0.029   
                 Number of Edges                          0.045 $\pm $ 0.025   
                 Number of Nodes                         -0.011 $\pm $ 0.007   

                                 